In [ ]:
## Dependencies
import sys, os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from textwrap import wrap
import requests
import time
import urllib.parse
import re
import json
import ast
from itertools import islice

from IPython.core.display import display, HTML

display(HTML("""
<style>
.output_scroll {
    overflow-y: scroll !important;
    max-height: 300px;
}
</style>
"""))



In [ ]:
## Load Author IDs data from Elements

## File to Load
file_path_1 = "data/2025_04-09 ORCID Adoption Report for Feinberg School of Medicine.csv"
file_path_2 = "data/2025_03-13 Elements Publication Data_all years.csv"
file_path_3 = "data/2025_03-13 Incites Publication Sources.csv"
 
# Read the CSV file and store into Pandas DataFrame
load_Feinberg_ORCID = pd.read_csv(file_path_1, encoding="ISO-8859-1")
load_Feinberg_Publications = pd.read_csv(file_path_2, encoding="utf-8")
load_Journal_Metrics = pd.read_csv(file_path_3, encoding="utf-8")

load_Feinberg_ORCID.head()


In [ ]:
def clean_column_names(dataframes):
    """
    Clean column names in each dataframe by removing any strange characters.

    Args:
    - dataframes (list of pandas.DataFrame): List of dataframes to clean.

    Returns:
    - None
    """
    for df in dataframes:
        if df is not None:  # Check if the dataframe is not None
            # Strip weird text from column headers
            df.columns = df.columns.str.strip("ï»¿")

## Call the function
clean_column_names([load_Feinberg_ORCID, load_Feinberg_Publications, load_Journal_Metrics])


In [ ]:

def filter_publications_last_10_years(df, date_column="Publication-Date"):
    """
    Filters the given DataFrame (df) for rows where the publication date is within the last 10 calendar years.
    
    The function attempts to convert the date in the `date_column` to a datetime object (using infer_datetime_format=True).
    Publications with unparseable dates (converted to NaT) are dropped from the filtered results.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame containing publication data.
        date_column (str): The name of the column containing publication dates.
                           Defaults to "Publication-Date".
    
    Returns:
        pd.DataFrame: A DataFrame filtered to only include publications from the last 10 calendar years.
    """
    # Work on a copy to avoid modifying the original DataFrame.
    df_filtered = df.copy()
    
    # Attempt to convert the date column to datetime.
    # If a value cannot be converted, it becomes NaT.
    df_filtered[date_column] = pd.to_datetime(df_filtered[date_column], errors='coerce', infer_datetime_format=True)
    
    # Get the current timestamp.
    current_time = pd.Timestamp.now()
    
    # Determine the cutoff date: January 1 of (current year - 10).
    cutoff_year = current_time.year - 10
    cutoff_date = pd.Timestamp(year=cutoff_year, month=1, day=1)
    
    # Filter out rows with invalid dates (NaT) and only keep rows on or after the cutoff date.
    filtered_df = df_filtered[df_filtered[date_column] >= cutoff_date]
    
    return filtered_df


filtered_publications = filter_publications_last_10_years(load_Feinberg_Publications)
filtered_publications.head()


In [ ]:

def merge_feinberg_data(feinberg_orcid, feinberg_publications):
    """
    Merges Feinberg_ORCID with Feinberg_Publications on matching NetIDs.
    
    Feinberg_ORCID should contain a column named 'NetID' and 
    Feinberg_Publications should contain a column named 'Net ID'.
    
    After merging, the following columns are dropped from the resulting dataframe:
    'Position', 'Department', 'School', 'Employee ID', 'Group ID', 'User ID'
    
    Optionally, the duplicate 'Net ID' column (from Publications) is removed.
    
    Parameters:
      feinberg_orcid (pd.DataFrame): DataFrame containing ORCID data with a 'NetID' column.
      feinberg_publications (pd.DataFrame): DataFrame containing Publications data with a 'Net ID' column.
    
    Returns:
      pd.DataFrame: A merged dataframe with unwanted columns removed.
    """
    # Merge the dataframes on the matching NetID columns.
    merged_df = feinberg_orcid.merge(
        feinberg_publications,
        left_on='NetID',
        right_on='Net ID',
        how='left'
    )
    
    # Define columns to drop from the Publications data.
    columns_to_drop = ['Position_y', 'Department', 'School', 'Employee ID', 'Group ID', 'User ID']
    
    # Drop unwanted columns; ignore errors if any column is missing.
    merged_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    
    # Optionally, drop the duplicate 'Net ID' column.
    merged_df.drop(columns=['Net ID'], inplace=True, errors='ignore')

        
    # Fill in missing DOIs: if 'DOI' is NaN but 'CrossRef' has a value, use the 'CrossRef' value.
    if 'doi' in merged_df.columns and 'Crossref' in merged_df.columns:
        merged_df['doi'] = merged_df['doi'].fillna(merged_df['Crossref'])
    
    return merged_df


merged_data = merge_feinberg_data(load_Feinberg_ORCID, filtered_publications)

print(merged_data.columns)

In [ ]:
department_filter = ['Preventive Medicine', 'Medical Education']

filtered_department_df = merged_data[
    merged_data['Department (primary appointment)'].isin(department_filter)
].copy()

# (optional) reset the index if you like
filtered_department_df.reset_index(drop=True, inplace=True)

filtered_department_df.head()

In [ ]:

def analyze_missing_doi(df):
    """
    Analyzes a DataFrame to determine how many rows have an empty DOI,
    and returns a summary of the corresponding 'Type' and 'Types' values.
    
    In this function, an empty DOI is defined as either a NaN value or a blank string.
    
    Parameters:
        df (pd.DataFrame): DataFrame that contains at least 'DOI', 'Type', and 'Types' columns.
    
    Returns:
        missing_count (int): The number of rows with missing DOI.
        grouped_summary (pd.DataFrame): A DataFrame summarizing the missing DOI rows
                                        grouped by 'Type' and 'Types' with a count for each.
    """
    # Create a mask: consider DOI as missing if it is NaN or equal to an empty string.
    missing_mask = df['doi'].isna() | (df['doi'] == '')
    
    # Subset the DataFrame to just the rows missing DOI.
    missing_df = df[missing_mask]
    
    # Count the total missing rows.
    missing_count = missing_df.shape[0]
    
    # Group by 'Type' and 'Types' columns if both exist:
    if ('Type' in missing_df.columns) and ('Types' in missing_df.columns):
        grouped_summary = missing_df.groupby(['Type', 'Types']).size().reset_index(name='MissingCount')
    elif 'Type' in missing_df.columns:
        grouped_summary = missing_df.groupby('Type').size().reset_index(name='MissingCount')
    elif 'Types' in missing_df.columns:
        grouped_summary = missing_df.groupby('Types').size().reset_index(name='MissingCount')
    else:
        grouped_summary = pd.DataFrame()  # Return an empty DataFrame if none exist.
    
    return missing_count, grouped_summary


missing_count, summary_doi_df = analyze_missing_doi(filtered_department_df)
print("Missing DOI count:", missing_count)
summary_doi_df.head(50)


In [ ]:
def calculate_missing_percent_by_type(missing_count, summary_df):
    """
    Calculates the percentage of missing DOIs for each "Type" based on a summary DataFrame.
    
    The summary_df is expected to have at least the columns "Type" and "MissingCount".
    The function groups the missing counts by "Type" and then calculates the percentage of
    the overall missing DOI count that each type accounts for.
    
    Parameters:
        missing_count (int): Total number of rows with a missing DOI.
        summary (pd.DataFrame): A DataFrame that includes the columns "Type" and "MissingCount".
        
    Returns:
        pd.DataFrame: A DataFrame with columns:
           - "Type": The publication type.
           - "TotalMissing": The sum of missing DOI counts for that type.
           - "MissingPercentage": The percentage (of overall missing_count) for that type.
             This is calculated as (TotalMissing / missing_count) * 100.
    """
    # Check that the required columns exist.
    if 'Type' not in summary_df.columns:
        raise ValueError("The summary_df must contain 'Type' column.")
    
    # Group by "Type" and sum the MissingCount if needed.
    grouped = summary_df.groupby('Type', as_index=False)['MissingCount'].sum()
    grouped.rename(columns={'MissingCount': 'TotalMissing'}, inplace=True)
    
    # Calculate the missing percentage for each type.
    grouped['MissingPercentage'] = grouped['TotalMissing'] / missing_count * 100
    
    # Optionally, sort by MissingPercentage descending.
    grouped = grouped.sort_values(by='MissingPercentage', ascending=False).reset_index(drop=True)
    
    return grouped




type_missing_df = calculate_missing_percent_by_type(missing_count, summary_doi_df)
type_missing_df.head()


In [ ]:
#### Query OpenAlex APIs based on Elements DOIs

def build_doi_filter(doi_list):
    """
    Builds the OpenAlex filter string by joining all DOIs with '|',
    and then prefixing the entire string once with 'doi:'.
    
    doi_list can contain either:
      - short DOIs like "10.1371/journal.pone.0266781"
      - or full URLs like "https://doi.org/10.1371/journal.pone.0266781"
    """
    # Clean whitespace
    cleaned = [str(doi).strip() for doi in doi_list]
    # Join once and prefix
    return "doi:" + "|".join(cleaned)

def query_openalex_group(doi_chunk, session, mailto="karen.gutzman@northwestern.edu"):
    """
    Queries OpenAlex for up to 100 DOIs at once using the OR syntax.
    
    Parameters:
      doi_chunk (list of str): Up to 100 DOI strings.
      session (requests.Session): A persistent HTTP session.
      mailto (str): Your email for API performance.
    """
    base_url = "https://api.openalex.org/works"
    filter_value = build_doi_filter(doi_chunk)
    params = {
        "filter": filter_value,
        "per-page": len(doi_chunk),  # per-page=100 at most
        "mailto": mailto
    }
    try:
        resp = session.get(base_url, params=params)
        # ←— print the actual URL with params encoded
        #print("Request URL:", resp.url) 
        #print("Request URL:", urllib.parse.unquote(resp.url))
        # print("Status:", resp.status_code)
        # print("Payload keys:", resp.json().keys())
        # print("First item:", resp.json().get("results", [])[:1])
        resp.raise_for_status()
        
        return resp.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying DOIs {doi_chunk}: {e}")
        return None

def query_unique_dois(df, doi_column="doi",
                      mailto="karen.gutzman@northwestern.edu",
                      group_size=100, delay=0.1):
    """
    Extracts unique DOIs, batches them into groups of up to 100, 
    queries OpenAlex once per group, and prints progress.
    
    Returns:
      doi_results: dict mapping each DOI → its JSON result or None.
      errors:      list of DOIs that failed.
    """
    # 1) Extract and clean unique DOIs
    unique = df[doi_column].dropna().unique()
    unique = [str(x).strip() for x in unique]
    
    total = len(unique)
    total_groups = (total + group_size - 1) // group_size
    
    doi_results = {}
    errors = []
    session = requests.Session()
    
    for grp in range(total_groups):
        start = grp * group_size
        chunk = unique[start:start + group_size]
        print(f"Processing group {grp+1}/{total_groups} "
              f"(DOIs {start+1}–{start+len(chunk)} of {total})…")
        
        data = query_openalex_group(chunk, session, mailto=mailto)
        if data is None:
            # whole chunk failed → mark each DOI as error
            for doi in chunk:
                doi_results[doi] = None
                errors.append(doi)
        else:
            # map returned results to DOI
            mapping = {}
            for work in data.get("results", []):
                # OpenAlex gives you something like "https://doi.org/10.xxxx/abcd"
                full = work.get("doi", "").strip().lower()
                # strip off the URL prefix so we also match "10.xxxx/abcd"
                short = full.replace("https://doi.org/", "")
                mapping[full]  = work
                mapping[short] = work
            for doi in chunk:
                for doi in chunk:
                    key = doi.strip().lower()
                    doi_results[doi] = mapping.get(key)
                    if doi_results[doi] is None:
                        errors.append(doi)
        
        time.sleep(delay)
    
    session.close()
    return doi_results, errors

def attach_openalex_results(df, doi_results, doi_column="doi"):
    """
    Adds an 'openalex_data' column to df by mapping each row's DOI to doi_results.
    """
    out = df.copy()
    out['openalex_data'] = out[doi_column].astype(str).str.strip().map(doi_results)
    return out

# ——————————————————————————————
# Example usage:

# merged_data = pd.read_csv("…")   # your dataframe with a 'doi' column
doi_results, error_list = query_unique_dois(
    filtered_department_df,
    doi_column="doi",
    mailto="karen.gutzman@northwestern.edu",
    group_size=100,
    delay=0.1
)

# Attach back to the full DataFrame
merged_with_api = attach_openalex_results(filtered_department_df, doi_results, doi_column="doi")

# If you want the errors as a DataFrame:
error_df = pd.DataFrame({"doi": error_list})

print(merged_with_api.head())
print("Errors:", error_df.head())


In [ ]:
merged_with_api.to_csv("output/merged_with_api_department.csv", index=False)
error_df.to_csv("output/error_df_department.csv", index=False)

In [ ]:
### Retry problem DOIs


def sanitize_doi(raw):
    """
    Trim whitespace and drop trailing commas/semicolons.
    """
    return raw.strip().rstrip(';, ')


def extract_valid_doi(s):
    """
    Extract the core DOI (10.xxxx/…) from a messy string.
    Returns None if no valid DOI pattern is found.
    """
    match = re.search(r"(10\.\d{4,9}/[^\s'\";,|]+)", s)
    return match.group(1) if match else None


def clean_core_doi(raw):
    """
    Sanitize raw input, extract valid DOI, strip trailing junk,
    and optionally skip book-chapter DOIs (B978-...).
    Returns None for unfixable or excluded DOIs.
    """
    s = sanitize_doi(raw)
    core = extract_valid_doi(s)
    if not core:
        return None

    # Remove only trailing punctuation: periods, parentheses, hash
    core = re.sub(r"[)\.#]+$", "", core)

    # Optionally skip book-chapter DOIs (example: B978-...)
    if core.upper().startswith("10.1016/B978"):
        return None

    return core


def retry_failed_dois(error_df,
                      doi_results,
                      mailto="karen.gutzman@northwestern.edu",
                      group_size=100,
                      delay=0.1):
    """
    Retries only the DOIs in error_df['doi'], cleans them,
    batches them in groups, and updates doi_results in place.
    Returns a list of DOIs that still failed after cleaning.
    """
    # 1) Clean and dedupe
    raw_list = error_df['doi'].dropna().astype(str).tolist()
    cleaned = []
    for raw in raw_list:
        core = clean_core_doi(raw)
        if core:
            cleaned.append(core)
        else:
            print(f"  ⚠ Skipping unfixable DOI: {raw!r}")
    to_retry = list(dict.fromkeys(cleaned))

    session = requests.Session()
    base_url = "https://api.openalex.org/works"
    still_errors = []

    total = len(to_retry)
    total_groups = (total + group_size - 1) // group_size

    for grp in range(total_groups):
        start = grp * group_size
        chunk = to_retry[start:start + group_size]
        print(f"Retrying group {grp+1}/{total_groups} "
              f"(DOIs {start+1}–{start+len(chunk)} of {total})…")

        params = {
            "filter": "doi:" + "|".join(chunk),
            "per-page": len(chunk),
            "mailto": mailto
        }

        try:
            resp = session.get(base_url, params=params)
            resp.raise_for_status()
            results = resp.json().get("results", [])
        except requests.exceptions.RequestException as e:
            print(f"  ↻ Chunk failed again: {e}")
            still_errors.extend(chunk)
            time.sleep(delay)
            continue

        # Build mapping for work lookups
        mapping = {}
        for work in results:
            doi_url = work.get("doi", "").strip().lower()
            short = doi_url.replace("https://doi.org/", "")
            mapping[doi_url] = work
            mapping[short] = work

        # Update doi_results and collect failures
        for doi in chunk:
            record = mapping.get(doi.lower())
            doi_results[doi] = record or None
            if record is None:
                still_errors.append(doi)

        time.sleep(delay)

    session.close()
    return still_errors


# ————— Usage Example —————
# error_df    = pd.DataFrame({"doi": error_list})
# doi_results = {doi: None for doi in error_list}

still_errors = retry_failed_dois(
    error_df,
    doi_results,
    mailto="karen.gutzman@northwestern.edu",
    group_size=100,
    delay=0.1
)

# Create DataFrame for DOIs that still failed
error_2_df = pd.DataFrame({"doi": still_errors})
print("DOIs still failing after retry:", still_errors)
print(error_2_df)

# If needed, re-attach to original DataFrame:
# merged_with_api = attach_openalex_results(merged_data, doi_results, doi_column="doi")


In [ ]:
merged_with_api.to_csv("output/merged_with_api_department.csv", index=False)
error_2_df.to_csv("output/error_2_df_department.csv", index=False)

In [ ]:
## File to Load
file_path_3 = "output/merged_with_api_department.csv"

 
# Read the CSV file and store into Pandas DataFrame
merged_with_api = pd.read_csv(file_path_3, encoding="ISO-8859-1")
merged_with_api.head()

In [ ]:
# merged_with_api.columns

# # ─── Examples: Print a specific cell given row index and column name ───
# # Suppose you want the value in row index 10 of the "OpenAlexID" column:
# idx = 10
# col = 'openalex_data'
# print("Value at row", idx, "column", col, ":", merged_with_api.loc[idx, col])

In [ ]:
#### Make columns for data in OpenAlex JSON

def ensure_dict(x):
    # if it already is a dict, leave it alone
    if isinstance(x, dict):
        return x
    # if it’s a JSON string, try json.loads
    if isinstance(x, str):
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            # maybe it’s a Python repr
            try:
                return ast.literal_eval(x)
            except Exception:
                return {}
    return {}

# apply to get a clean dict column
merged_with_api['openalex_parsed'] = merged_with_api['openalex_data'].apply(ensure_dict)

# now extract the pieces you asked for
merged_with_api['OpenAlexID'] = merged_with_api['openalex_parsed'].apply(
    lambda d: d.get('ids', {}).get('openalex')
)

merged_with_api['Related_Works'] = merged_with_api['openalex_parsed'].apply(
    lambda d: d.get('related_works', [])
)

merged_with_api['Referenced_Works'] = merged_with_api['openalex_parsed'].apply(
    lambda d: d.get('referenced_works', [])
)

merged_with_api['is_oa'] = merged_with_api['openalex_parsed'].apply(
    lambda d: d.get('open_access', {}).get('is_oa')
)
merged_with_api['oa_status'] = merged_with_api['openalex_parsed'].apply(
    lambda d: d.get('open_access', {}).get('oa_status')
)

# inspect
merged_with_api[['OpenAlexID','Related_Works','Referenced_Works','is_oa','oa_status']].head()


In [ ]:
## Analyze missing data from OpenAlex JSON

cols = ['OpenAlexID','Related_Works','Referenced_Works','is_oa','oa_status']
summary = {}

for col in cols:
    s = merged_with_api[col]
    # count rows where the entry is a list of length 0
    n_empty_list = s.apply(lambda x: isinstance(x, list) and len(x)==0).sum()
    # count rows where the entry is NaN or None
    n_missing    = s.isna().sum()
    summary[col] = {'empty_list': n_empty_list, 'missing': n_missing}

summary_2_df = pd.DataFrame(summary).T

print(len(merged_with_api))
summary_2_df.head()


In [ ]:
### Analyze missing data from OpenAlex JSON by year of publication

# 1. Make sure your dates are datetime
df = merged_with_api.copy()
df['Publication-Date'] = pd.to_datetime(df['Publication-Date'], errors='coerce')

# 2. Pull out the year
df['pub_year'] = df['Publication-Date'].dt.year

# 3. Define the columns you want to summarize
cols = ['OpenAlexID','Related_Works','Referenced_Works','is_oa','oa_status']

# 4. A helper to compute empty_list & missing for one group
def summarize_group(g):
    out = {}
    for c in cols:
        s = g[c]
        out[(c, 'empty_list')] = s.apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
        out[(c, 'missing')]    = s.isna().sum()
    return pd.Series(out)

# 5. Group by year and apply
summary_by_year = (
    df.groupby('pub_year')
      .apply(summarize_group)
)

# 6. (Optional) flatten the MultiIndex columns
summary_by_year.columns = [
    f"{col}_{stat}" for col, stat in summary_by_year.columns
]

# Inspect the result
summary_by_year.head(20)


In [ ]:
# #merged_with_api.columns

# # ─── Examples: Print a specific cell given row index and column name ───
# # Suppose you want the value in row index 10 of the "OpenAlexID" column:
# idx = 10
# col = 'OpenAlexID'
# print("Value at row", idx, "column", col, ":", merged_with_api.loc[idx, col])

In [ ]:
# Save with gzip compression
merged_with_api.to_pickle("output/merged_with_api_department_2.pkl", compression="gzip")

# Load it back (pandas infers compression from extension)
merged_with_api_df = pd.read_pickle("output/merged_with_api_department_2.pkl", compression="gzip")
merged_with_api_df.head()

In [ ]:
### Query OpenAlex to get Cited_By URLS 


def query_cited_by_group(alex_chunk, session, mailto="karen.gutzman@northwestern.edu"):
    """
    Queries OpenAlex for up to 100 OpenAlex IDs at once using the "cites" filter.

    Parameters:
      alex_chunk (list of str): Up to 100 OpenAlex work URLs or IDs (e.g. "https://openalex.org/W1234567890").
      session (requests.Session): A persistent HTTP session.
      mailto (str): Email for API performance.

    Returns:
      dict: JSON response if successful, None on error.
    """
    base_url = "https://api.openalex.org/works"

    # Normalize IDs: strip full URL to just the ID portion
    ids = []
    for a in alex_chunk:
        if a.startswith("http"):
            ids.append(a.rstrip("/").split("/")[-1])
        else:
            ids.append(a)

    # Build filter string: cites:W123|W456|...
    filter_value = "cites:" + "|".join(ids)
    params = {"filter": filter_value, "per-page": len(ids), "mailto": mailto}

    # Debug: show exact URL being requested
    debug_url = f"{base_url}?{urllib.parse.urlencode(params)}"
    print("Debug query_cited_by_group URL ->", debug_url)

    try:
        resp = session.get(base_url, params=params)
        print("Received response status", resp.status_code)
        print("Resolved URL ->", resp.url)
        resp.raise_for_status()
        return resp.json()
    except requests.exceptions.RequestException as e:
        print(f"Error querying cited-by for {ids}: {e}")
        return None


def query_unique_cited_by(df, alex_column="OpenAlexID",
                          mailto="karen.gutzman@northwestern.edu",
                          group_size=100, delay=0.1):
    """
    Extracts unique OpenAlex IDs (raw), batches into groups, queries OpenAlex
    "cites" filter to retrieve works citing them.

    Returns:
      cited_results: dict mapping raw OpenAlexID -> list of citing works or None
      errors: list of raw IDs that failed
    """
    # Normalize the column to raw IDs, drop NaN, dedupe
    raw_ids = (
        df[alex_column]
        .dropna()
        .astype(str)
        .str.strip()
        .apply(lambda x: x.rstrip("/").split("/")[-1])
        .drop_duplicates()
        .tolist()
    )
    total = len(raw_ids)
    print(f"Debug query_unique_cited_by: extracted {total} unique OpenAlex IDs from column '{alex_column}'.")
    if total == 0:
        print("No IDs to query. Check that your dataframe and column name are correct.")
        return {}, []

    total_groups = (total + group_size - 1) // group_size

    cited_results = {}
    errors = []
    session = requests.Session()

    for grp in range(total_groups):
        start = grp * group_size
        chunk = raw_ids[start:start + group_size]
        print(f"Processing group {grp+1}/{total_groups} "
              f"(IDs {start+1}–{start+len(chunk)} of {total})… IDs: {chunk}")

        data = query_cited_by_group(chunk, session, mailto=mailto)
        if data is None:
            print(f"Chunk {grp+1} failed entirely, marking IDs as error: {chunk}")
            for rid in chunk:
                cited_results[rid] = None
                errors.append(rid)
        else:
            results = data.get("results", [])
            mapping = {rid: [] for rid in chunk}
            for work in results:
                for ref_url in work.get("referenced_works", []):
                    ref_id = ref_url.rstrip("/").split("/")[-1]
                    if ref_id in mapping:
                        mapping[ref_id].append(work)
            for rid in chunk:
                cited_results[rid] = mapping.get(rid, [])

        time.sleep(delay)

    session.close()
    return cited_results, errors


def attach_cited_by_results(df, cited_results, alex_column="OpenAlexID"):
    """
    Adds a 'cited_by_data' column to df by mapping each row's OpenAlexID to its citing works.
    """
    out = df.copy()
    out['cited_by_data'] = (
        out[alex_column]
        .astype(str)
        .str.strip()
        .apply(lambda x: x.rstrip("/").split("/")[-1])
        .map(cited_results)
    )
    return out


cited_results, error_3_list = query_unique_cited_by(
    merged_with_api,
    alex_column="OpenAlexID",
    mailto="karen.gutzman@northwestern.edu",
    group_size=100,
    delay=0.1
)
error_3_df = pd.DataFrame({"OpenAlexID": error_3_list})
merged_with_citations = attach_cited_by_results(
    merged_with_api,
    cited_results,
    alex_column="OpenAlexID"
)
print(merged_with_citations[['OpenAlexID', 'cited_by_data']].head())


In [ ]:
# # How many IDs got mapped to a non‑empty list?
# nonempty = sum(bool(v) for v in cited_results.values())
# print(f"{nonempty} IDs have ≥1 citing works out of {len(cited_results)} total.")

# # Inspect the first few entries
# import itertools, pprint
# for rid, works in itertools.islice(cited_results.items(), 5):
#     pprint.pprint((rid, works))

In [ ]:
##### Query OpenAlex to get Journal Information
from time import sleep

# STEP 1: Prepare your data
issns = merged_with_citations['ISSN'].dropna().unique().tolist()
missing_issn_df = merged_with_citations[merged_with_citations['ISSN'].isna()].copy()

# STEP 2: Chunk ISSNs into batches of 100
batch_size = 100
issn_batches = [issns[i:i + batch_size] for i in range(0, len(issns), batch_size)]

# STEP 3: Function to fetch OpenAlex journal metadata for a batch
def fetch_openalex_journal_data(issn_batch, mailto="karen.gutzman@northwestern.edu"):
    filter_query = '|'.join(issn_batch)
    url = f"https://api.openalex.org/journals?filter=issn:{filter_query}&per-page=100&mailto={mailto}"
    print("Requesting:", url)  # Optional: to debug
    response = requests.get(url)
    if response.status_code == 200:
        return response.json().get('results', [])
    else:
        print(f"⚠️ Request failed with status code {response.status_code}")
        return []


# STEP 4: Loop over batches and collect results
all_results = []
for i, batch in enumerate(issn_batches):
    print(f"🔄 Fetching batch {i+1}/{len(issn_batches)}...")
    batch_results = fetch_openalex_journal_data(batch, mailto="karen.gutzman@northwestern.edu")
    all_results.extend(batch_results)
    sleep(0.1)  # Be polite to the API!

# STEP 5: Normalize results into a DataFrame
openalex_journals_df = pd.json_normalize(all_results)

# Optional: Keep only the fields you care about
openalex_journals_df = openalex_journals_df[[
    'id', 'display_name', 'issn_l', 'is_oa', 'is_in_doaj', 'host_organization_name', 'host_organization'
]].rename(columns={
    'id': 'OpenAlexJournalID',
    'display_name': 'Journal Name',
    'issn_l': 'ISSN',
    'is_oa': 'Journal_is_oa',
    'is_in_doaj': 'Journal_is_in_doaj',
    'host_organization_name': 'Journal Publisher',
    'host_organization': 'Host Organization'
})

# STEP 6: View a sample
openalex_journals_df.head()

# STEP 7: Save for future merging or reference
openalex_journals_df.to_csv("output/openalex_journal_metadata.csv", index=False)
missing_issn_df.to_csv("output/journals_missing_issn.csv", index=False)


In [ ]:
### Query OpenAlex API with Cited_by URLS to get Citing Document Data

# —————— helper functions ——————

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        batch = list(islice(it, size))
        if not batch:
            break
        yield batch

def fetch_journal_info_batch(
    citing_urls,
    mailto="karen.gutzman@northwestern.edu",
    delay=0.1,
    batch_size=100
):
    session  = requests.Session()
    base_url = "https://api.openalex.org/works"
    all_rows = []

    for batch_num, urls in enumerate(chunked(citing_urls, batch_size), 1):
        ids = [u.rstrip("/").split("/")[-1] for u in urls]
        filter_val = "openalex:" + "|".join(ids)

        print(f"[Batch {batch_num}] querying {len(ids)} IDs…")
        resp = session.get(base_url, params={
            "filter":   filter_val,
            "per-page": len(ids),
            "mailto":   mailto,
        })
        resp.raise_for_status()
        results = resp.json().get("results", [])

        for work in results:
            work_id   = work["id"].split("/")[-1]
            pub_date  = work.get("publication_date")
            doi       = work.get("doi")
            work_type = work.get("type")
            for loc in work.get("locations", []):
                src = loc.get("source") or {}
                if src.get("type") == "journal":
                    all_rows.append({
                        "OpenAlexID":       work_id,
                        "publication_date": pub_date,
                        "doi":              doi,
                        "work_type":        work_type,
                        "journal_name":     src.get("display_name"),
                        "issn_l":           src.get("issn_l"),
                        "issn_list":        src.get("issn", []),
                    })

        time.sleep(delay)

    session.close()
    return pd.DataFrame(all_rows)



# 1. Flatten all 'id' fields out of your cited_by_data column
all_urls = []
for entry in merged_with_citations['cited_by_data']:
    if isinstance(entry, list):
        all_urls.extend(item.get('id') for item in entry if 'id' in item)

# 2. Drop None/NaN and dedupe
citing_urls = list({url for url in all_urls if url})

print(f"Found {len(citing_urls)} unique citing URLs.")

# —————— Now call the batch fetch ——————
df_journals = fetch_journal_info_batch(
    citing_urls,
    mailto="karen.gutzman@northwestern.edu",
    delay=0.1,
    batch_size=100
)

print(df_journals.head())


In [ ]:
merged_with_citations.columns

In [ ]:
# Save with gzip compression
df_journals.to_pickle("output/df_journals_department.pkl", compression="gzip")

# Load it back (pandas infers compression from extension)
citing_journals_df = pd.read_pickle("output/df_journals_department.pkl", compression="gzip")

In [ ]:
################ Query OpenAlex with References_Works URLs #################

import requests
import pandas as pd
import time
from itertools import islice

# —————— helper functions ——————

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        batch = list(islice(it, size))
        if not batch:
            break
        yield batch

def fetch_journal_info_batch(
    reference_urls,
    mailto="karen.gutzman@northwestern.edu",
    delay=0.1,
    batch_size=100
):
    session  = requests.Session()
    base_url = "https://api.openalex.org/works"
    all_rows = []

    for batch_num, urls in enumerate(chunked(reference_urls, batch_size), 1):
        ids = [u.rstrip("/").split("/")[-1] for u in urls]
        filter_val = "openalex:" + "|".join(ids)

        print(f"[Batch {batch_num}] querying {len(ids)} IDs…")
        resp = session.get(base_url, params={
            "filter":   filter_val,
            "per-page": len(ids),
            "mailto":   mailto,
        })
        resp.raise_for_status()
        results = resp.json().get("results", [])

        for work in results:
            work_id   = work["id"].split("/")[-1]
            pub_date  = work.get("publication_date")
            doi       = work.get("doi")
            work_type = work.get("type")
            for loc in work.get("locations", []):
                src = loc.get("source") or {}
                if src.get("type") == "journal":
                    all_rows.append({
                        "OpenAlexID":       work_id,
                        "publication_date": pub_date,
                        "doi":              doi,
                        "work_type":        work_type,
                        "journal_name":     src.get("display_name"),
                        "issn_l":           src.get("issn_l"),
                        "issn_list":        src.get("issn", []),
                    })

        time.sleep(delay)

    session.close()
    return pd.DataFrame(all_rows)



# 1. Flatten all URLs out of your Referenced_Works column
all_urls = []
for entry in merged_with_citations['Referenced_Works']:
    if isinstance(entry, list):
        # entry is a list of URL strings—just extend it
        all_urls.extend(entry)

# 2. Drop None/NaN (if any slipped through) and dedupe
reference_urls = [u for u in set(all_urls) if isinstance(u, str)]

# 3. Sanity check
print(f"Found {len(reference_urls)} unique reference URLs.")
print("Sample URLs:", reference_urls[:5])

# 4. Batch‐call the API
reference_journals_df = fetch_journal_info_batch(
    reference_urls,
    mailto="karen.gutzman@northwestern.edu",
    delay=0.1,
    batch_size=100
)

print(reference_journals_df.head())

In [ ]:
# Save with gzip compression
reference_journals_df.to_pickle("output/reference_journals_df.pkl", compression="gzip")

# Load it back (pandas infers compression from extension)
reference_journals_df = pd.read_pickle("output/reference_journals_df.pkl", compression="gzip")

In [ ]:
# Save with gzip compression
reference_journals_df.to_pickle("output/reference_journals_df_department.pkl", compression="gzip")

# Load it back (pandas infers compression from extension)
reference_journals_df = pd.read_pickle("output/reference_journals_df.pkl", compression="gzip")

In [ ]:
### Filter the journal metrics dataframe

# Step 1: Filter only rows where JIF Percentile is not null
valid_metrics = load_Journal_Metrics.dropna(subset=['Average JIF Percentile'])

# Step 2: Get the latest year with actual data
latest_year = valid_metrics['Publication Year'].max()

# Step 3: Filter to only include rows from that year
latest_journal_metrics_df = load_Journal_Metrics[load_Journal_Metrics['Publication Year'] == latest_year].copy()

# Step 4: Select relevant columns
columns_to_keep = [
    'ISSN', 'eISSN', 'Name', 'Publisher (all)', 'Publisher (unified)', 
    'Publication Year', 'Journal Impact Factor', 'Average JIF Percentile',
    'Journal Citation Indicator', 'WoS Categories', 'JIF Quartile',
    'JCI Quartile', 'JCI Percentile', 'Source Type'
]

latest_journal_metrics_df = latest_journal_metrics_df[columns_to_keep]
latest_journal_metrics_df = latest_journal_metrics_df.sort_values(by='Name').reset_index(drop=True)


In [ ]:
## Review dataframes
print("publications_by_author_df columns below")
print(merged_with_api_df.columns)
print("journals_from_authors_reference_lists_df columns below")
print(reference_journals_df.columns)
print("journals_from_citing_documents_df columns below")
print(citing_journals_df.columns)
print("journal_metrics_data_by_year_df columns below")
print(latest_journal_metrics_df.columns)

In [ ]:
# # Cleaned and polished version of your journal scoring pipeline
# Clean ISSN fields
for df in [merged_with_api_df, citing_journals_df, reference_journals_df]:
    if 'ISSN' in df.columns:
        df['ISSN'] = df['ISSN'].astype(str).str.strip().str.lower()
    if 'issn_l' in df.columns:
        df['issn_l'] = df['issn_l'].astype(str).str.strip().str.lower()

# Ensure every publication has some kind of ISSN
if 'issn_l' in merged_with_api_df.columns:
    merged_with_api_df['ISSN'] = merged_with_api_df['ISSN'].fillna(merged_with_api_df['issn_l'])

# ---- FILTER DEPARTMENTS ----

departments_of_interest = ['Medical Education', 'Preventive Medicine']
pubs_df = merged_with_api_df[merged_with_api_df['Department (primary appointment)'].isin(departments_of_interest)].copy()

# ---- AUTHORSHIP ----
pubs_df['Total_Authors'] = 5
nu_authorship_counts = pubs_df.groupby(['Publication ID', 'Department (primary appointment)']).size().reset_index(name='NU_Authors')
pubs_df = pubs_df.merge(nu_authorship_counts, on=['Publication ID', 'Department (primary appointment)'], how='left')
pubs_df['Fractional_Contribution'] = pubs_df['NU_Authors'] / pubs_df['Total_Authors']

fractional_authorship = pubs_df.groupby(['ISSN', 'Department (primary appointment)'])['Fractional_Contribution'].sum().reset_index().rename(columns={'Fractional_Contribution': 'NU_Author_Share'})

# ---- CITATIONS ----
citations_to_nu = citing_journals_df.groupby('issn_l').size().reset_index(name='CitationsToNU')
citations_of = reference_journals_df.groupby('issn_l').size().reset_index(name='CitationsByNU')

# ---- JOURNAL METRICS ----
wos_rank = latest_journal_metrics_df.groupby('ISSN')['Average JIF Percentile'].max().reset_index().rename(columns={'Average JIF Percentile': 'WOSPercentile'})

# ---- OA ESTIMATION ----
oa_ratio_by_journal = pubs_df.groupby('ISSN')['is_oa'].mean().reset_index().rename(columns={'is_oa': 'OA_Ratio'})
openalex_journals_df['ISSN'] = openalex_journals_df['ISSN'].str.strip().str.lower()
oa_ratio_by_journal = oa_ratio_by_journal.merge(openalex_journals_df[['ISSN', 'Journal_is_oa', 'Journal_is_in_doaj']], on='ISSN', how='left')

def classify_oa_type(row):
    if row.get('Journal_is_in_doaj') is True:
        return 'Gold'
    elif row['OA_Ratio'] >= 0.25:
        return 'Hybrid'
    else:
        return 'Subscription'

oa_ratio_by_journal['OA_Status_Estimated'] = oa_ratio_by_journal.apply(classify_oa_type, axis=1)

oa_score_map = {'Gold': 0, 'Hybrid': 5, 'Subscription': 10}
oa_ratio_by_journal['OA_Score'] = oa_ratio_by_journal['OA_Status_Estimated'].map(oa_score_map)

# ---- DEPT & INTERDISCIPLINARY ----
dept_match = pubs_df.groupby(['ISSN', 'Department (primary appointment)']).size().reset_index(name='DeptMatch')
dept_match['DeptMatch_Score'] = 10

interdisciplinary = pubs_df.groupby('ISSN')['Department (primary appointment)'].nunique().reset_index(name='DeptCount')
interdisciplinary['Interdisciplinary_Score'] = interdisciplinary['DeptCount'].apply(lambda x: 10 if x > 1 else 0)

# ---- CLEAN FOR MERGING ----
citations_to_nu = citations_to_nu.drop_duplicates(subset='issn_l')
citations_of = citations_of.drop_duplicates(subset='issn_l')
wos_rank = wos_rank.drop_duplicates(subset='ISSN')
dept_match = dept_match.drop_duplicates(subset=['ISSN', 'Department (primary appointment)'])
interdisciplinary = interdisciplinary.drop_duplicates(subset='ISSN')
journal_metadata = latest_journal_metrics_df[['ISSN', 'Name', 'Publisher (all)', 'Publisher (unified)']].drop_duplicates()

# ---- BUILD UNION BACKBONE ----
nu_journal_issns = set(pubs_df['ISSN'].dropna().unique())
ref_journal_issns = set(reference_journals_df['issn_l'].dropna().unique())
backbone_issns = pd.DataFrame({'ISSN': list(nu_journal_issns.union(ref_journal_issns))})

# ---- EXPAND TO ALL DEPT COMBOS ----
unique_departments = pubs_df['Department (primary appointment)'].dropna().unique()
backbone_df = pd.DataFrame([(issn, dept) for issn in backbone_issns['ISSN'] for dept in unique_departments], columns=['ISSN', 'Department (primary appointment)'])

# Merge all scoring inputs using backbone
journal_scores = backbone_df.merge(fractional_authorship, on=['ISSN', 'Department (primary appointment)'], how='left')
journal_scores['NU_Author_Share'] = journal_scores['NU_Author_Share'].fillna(0)

journal_scores = journal_scores.merge(citations_to_nu, left_on='ISSN', right_on='issn_l', how='left')
journal_scores = journal_scores.merge(citations_of, on='issn_l', how='left')
journal_scores = journal_scores.merge(wos_rank, on='ISSN', how='left')
journal_scores = journal_scores.merge(oa_ratio_by_journal[['ISSN', 'OA_Status_Estimated', 'OA_Score']], on='ISSN', how='left')
journal_scores = journal_scores.merge(dept_match[['ISSN', 'Department (primary appointment)', 'DeptMatch_Score']], on=['ISSN', 'Department (primary appointment)'], how='left')
journal_scores = journal_scores.merge(interdisciplinary[['ISSN', 'Interdisciplinary_Score']], on='ISSN', how='left')
journal_scores = journal_scores.merge(journal_metadata, on='ISSN', how='left')

# ---- BACKUP METRICS ----
jci_fallback = latest_journal_metrics_df[['ISSN', 'JCI Percentile']].drop_duplicates()
journal_scores = journal_scores.merge(jci_fallback, on='ISSN', how='left')

journal_scores['WOSPercentile'] = journal_scores['WOSPercentile'].fillna(0)
journal_scores['JCI Percentile'] = journal_scores['JCI Percentile'].fillna(0)
journal_scores['Effective_Percentile'] = journal_scores['WOSPercentile']
journal_scores.loc[journal_scores['Effective_Percentile'] == 0, 'Effective_Percentile'] = journal_scores['JCI Percentile']
journal_scores['Used_Fallback_Percentile'] = journal_scores['WOSPercentile'] == 0

# ---- REFERENCED ONLY FLAG ----
journal_scores['Referenced_Only'] = ~journal_scores['ISSN'].isin(nu_journal_issns)

# ---- FILL & SCORE ----
for col in ['CitationsToNU', 'CitationsByNU', 'OA_Score', 'DeptMatch_Score', 'Interdisciplinary_Score']:
    journal_scores[col] = journal_scores[col].fillna(0)

journal_scores['Authorship_Score'] = journal_scores.groupby('Department (primary appointment)')['NU_Author_Share'].transform(lambda x: 25 * (x / x.max()) if x.max() > 0 else 0)
journal_scores['CitationsTo_Score'] = 15 * (journal_scores['CitationsToNU'] / journal_scores['CitationsToNU'].max()) if journal_scores['CitationsToNU'].max() > 0 else 0
journal_scores['CitationsOf_Score'] = 10 * (journal_scores['CitationsByNU'] / journal_scores['CitationsByNU'].max()) if journal_scores['CitationsByNU'].max() > 0 else 0
journal_scores['WOS_Score'] = 20 * (journal_scores['Effective_Percentile'] / 100)

journal_scores['Total_Score'] = journal_scores[[
    'Authorship_Score', 'CitationsTo_Score', 'CitationsOf_Score',
    'WOS_Score', 'OA_Score', 'DeptMatch_Score', 'Interdisciplinary_Score'
]].sum(axis=1)

# ---- RECOMMENDATION ----
def recommendation(score):
    if score > 60:
        return "Keep / Fund"
    elif score > 50:
        return "Evaluate"
    else:
        return "Consider Cancellation"

journal_scores['Recommendation'] = journal_scores['Total_Score'].apply(recommendation)

# ---- REFERENCED ONLY FINAL RECOMMENDATION ----
def referenced_only_recommendation(row):
    if row['Referenced_Only']:
        if row['CitationsByNU'] > 10: #or row['WOS_Score'] > 10:
            return 'Keep (Cited)'
        else:
            return 'Low Value (Cited Only)'
    else:
        return row['Recommendation']

journal_scores['Recommendation_Final'] = journal_scores.apply(referenced_only_recommendation, axis=1)

# ---- FINAL DEDUPLICATION BY DEPARTMENT AND JOURNAL NAME ----
journal_scores = journal_scores.drop_duplicates(subset=['Department (primary appointment)', 'Name'])


# ---- OUTPUT ----
journal_scores.to_csv("output/journal_scoring_results.csv", index=False)
journal_scores.head(20)
